In [1]:
import subprocess
import sys
import importlib.metadata as metadata

packages = [
    "qiskit==2.2.3", "numpy==2.2.6", "scipy==1.15.3",
    "scikit-learn==1.6.1", "pandas==2.2.3", "matplotlib==3.10.1",
    "flask==3.1.0", "joblib==1.6.0", "jedi"
]

subprocess.check_call([sys.executable, "-m", "pip", "install", *packages])
subprocess.check_call([sys.executable, "-m", "pip", "check"])

print("Python:", sys.version.split()[0])
print({p.split("==")[0]: metadata.version(p.split("==")[0]) for p in packages})

Python: 3.13.18
{'qiskit': '2.2.3', 'numpy': '2.2.6', 'scipy': '1.15.3', 'scikit-learn': '1.6.1', 'pandas': '2.2.3', 'matplotlib': '3.10.1', 'flask': '3.1.0', 'joblib': '1.6.0', 'jedi': '0.20.1'}


In [16]:
import subprocess
import sys
import importlib.metadata as metadata
packages=['qiskit==2.2.3','numpy==2.2.6','scipy==1.15.3',
          'scikit-learn==1.6.1','pandas==2.2.3','matplotlib==3.10.1',
          'flask==3.1.0','joblib==1.6.0']
subprocess.check_call([sys.executable,'-m','pip','install',*packages])
subprocess.check_call([sys.executable,'-m','pip','check'])
print('Python:',sys.version.split()[0])
print({item.split('==')[0]:metadata.version(item.split('==')[0]) for item in packages})

Python: 3.13.18
{'qiskit': '2.2.3', 'numpy': '2.2.6', 'scipy': '1.15.3', 'scikit-learn': '1.6.1', 'pandas': '2.2.3', 'matplotlib': '3.10.1', 'flask': '3.1.0', 'joblib': '1.6.0'}


In [20]:
import numpy as np
import pandas as pd
from io import StringIO
from time import perf_counter
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, confusion_matrix
from qiskit.circuit.library import zz_feature_map, pauli_feature_map
from qiskit.quantum_info import Statevector

# source_row: scikit-learn 1.6.1 WDBC 데이터의 0부터 시작하는 행 번호
# 열 순서: source_row, radius, texture, perimeter, area, smoothness, compactness, label
# label: 0=원자료 양성(benign), 1=원자료 악성(malignant)
raw = """19,13.54,14.36,87.46,566.3,0.09779,0.08129,0
20,13.08,15.71,85.63,520,0.1075,0.127,0
21,9.504,12.44,60.34,273.9,0.1024,0.06492,0
37,13.03,18.42,82.61,523.8,0.08983,0.03766,0
46,8.196,16.84,51.71,201.9,0.086,0.05943,0
48,12.05,14.63,78.04,449.3,0.1031,0.09092,0
49,13.49,22.3,86.91,561,0.08752,0.07698,0
50,11.76,21.6,74.72,427.9,0.08637,0.04966,0
51,13.64,16.34,87.21,571.8,0.07685,0.06059,0
52,11.94,18.24,75.71,437.6,0.08261,0.04751,0
55,11.52,18.75,73.34,409,0.09524,0.05473,0
58,13.05,19.31,82.61,527.2,0.0806,0.03789,0
59,8.618,11.79,54.34,224.5,0.09752,0.05272,0
60,10.17,14.88,64.55,311.9,0.1134,0.08061,0
61,8.598,20.98,54.66,221.8,0.1243,0.08963,0
63,9.173,13.86,59.2,260.9,0.07721,0.08751,0
66,9.465,21.01,60.11,269.4,0.1044,0.07773,0
67,11.31,19.04,71.8,394.1,0.08139,0.04701,0
68,9.029,17.33,58.79,250.5,0.1066,0.1413,0
69,12.78,16.49,81.37,502.5,0.09831,0.05234,0
0,17.99,10.38,122.8,1001,0.1184,0.2776,1
1,20.57,17.77,132.9,1326,0.08474,0.07864,1
2,19.69,21.25,130,1203,0.1096,0.1599,1
3,11.42,20.38,77.58,386.1,0.1425,0.2839,1
4,20.29,14.34,135.1,1297,0.1003,0.1328,1
5,12.45,15.7,82.57,477.1,0.1278,0.17,1
6,18.25,19.98,119.6,1040,0.09463,0.109,1
7,13.71,20.83,90.2,577.9,0.1189,0.1645,1
8,13,21.82,87.5,519.8,0.1273,0.1932,1
9,12.46,24.04,83.97,475.9,0.1186,0.2396,1
10,16.02,23.24,102.7,797.8,0.08206,0.06669,1
11,15.78,17.89,103.6,781,0.0971,0.1292,1
12,19.17,24.8,132.4,1123,0.0974,0.2458,1
13,15.85,23.95,103.7,782.7,0.08401,0.1002,1
14,13.73,22.61,93.6,578.3,0.1131,0.2293,1
15,14.54,27.54,96.73,658.8,0.1139,0.1595,1
16,14.68,20.13,94.74,684.5,0.09867,0.072,1
17,16.13,20.68,108.1,798.8,0.117,0.2022,1
18,19.81,22.15,130,1260,0.09831,0.1027,1
22,15.34,14.26,102.5,704.4,0.1073,0.2135,1"""
df = pd.read_csv(StringIO(raw), header=None,
    names=['source_row','radius','texture','perimeter','area','smoothness','compactness','label'])
FEATURES = list(df.columns[1:7])
X = df[FEATURES].to_numpy(float)
y = df.label.to_numpy(int)
# 각 클래스의 앞 14개 학습, 뒤 6개 평가: 고정 분할
train = np.r_[0:14,20:34]
test = np.r_[14:20,34:40]
assert set(train).isdisjoint(test) and np.isfinite(X).all()
assert np.array_equal(np.bincount(y[train]), [14,14])
assert np.array_equal(np.bincount(y[test]), [6,6])

def prepare(n=2, indices=None):
    indices = train if indices is None else np.asarray(indices)
    standard = StandardScaler().fit(X[indices])
    pca = PCA(n_components=n, svd_solver='full').fit(standard.transform(X[indices]))
    angle = MinMaxScaler(feature_range=(0,1)).fit(pca.transform(standard.transform(X[indices])))
    def transform(values):
        # 학습 범위 밖 값은 0~1 rad 경계로 제한합니다.
        return np.clip(angle.transform(pca.transform(standard.transform(values))),0,1)
    transform.steps=(standard,pca,angle)
    return transform(X[indices]),transform(X[test]),transform,pca

def states(values, kind='ZZ', reps=1):
    n=values.shape[1]
    circuit=(zz_feature_map(n,reps=reps,entanglement='linear') if kind=='ZZ'
        else pauli_feature_map(n,reps=reps,entanglement='linear',paulis=['X','ZZ']))
    parameters=list(circuit.parameters)
    vectors=np.array([Statevector.from_instruction(circuit.assign_parameters(
        dict(zip(parameters,row)))).data for row in values])
    assert np.allclose(np.sum(abs(vectors)**2,axis=1),1)
    return vectors,circuit

def gram(a,b):
    return np.clip(abs(a.conj() @ b.T)**2,0,1)

def evaluate(a,b,labels,kind='ZZ',reps=1,weight=None):
    started=perf_counter()
    sa,circuit=states(a,kind,reps)
    sb,_=states(b,kind,reps)
    k=gram(sa,sa); kt=gram(sb,sa)
    kernel_seconds=perf_counter()-started
    assert np.allclose(k,k.T) and np.allclose(k.diagonal(),1)
    assert np.linalg.eigvalsh(k).min()>-1e-8
    models={'quantum':SVC(kernel='precomputed',C=1,class_weight=weight),
            'rbf':SVC(kernel='rbf',C=1,gamma='scale',class_weight=weight)}
    records=[]; predictions={}
    for name,model in models.items():
        t=perf_counter()
        model.fit(k if name=='quantum' else a,labels)
        pred=model.predict(kt if name=='quantum' else b)
        seconds=perf_counter()-t
        predictions[name]=pred
        records.append(dict(model=name,accuracy=accuracy_score(y[test],pred),
            balanced_accuracy=balanced_accuracy_score(y[test],pred),
            f1=f1_score(y[test],pred,zero_division=0),
            kernel_seconds=kernel_seconds if name=='quantum' else 0,
            fit_predict_seconds=seconds,
            total_seconds=seconds+(kernel_seconds if name=='quantum' else 0)))
        print(name,'confusion matrix [0,1] rows=true cols=predicted')
        print(confusion_matrix(y[test],pred,labels=[0,1]))
    return pd.DataFrame(records),models,sa,k,predictions,circuit

COMPARE=True

import joblib
import threading
from flask import Flask, request, jsonify, render_template_string
from werkzeug.serving import make_server
from urllib.request import urlopen

a,b,transform,pca=prepare(2)
sa,_=states(a)
k=gram(sa,sa)
models={'quantum':SVC(kernel='precomputed',C=1,probability=COMPARE,random_state=34)}
models['quantum'].fit(k,y[train])
if COMPARE:
    models['rbf']=SVC(kernel='rbf',C=1,gamma='scale',probability=True,random_state=34).fit(a,y[train])
bundle={'steps':transform.steps,'models':models,'training_states':sa,
        'features':FEATURES,'version':'wdbc_fixed_40_v2' if COMPARE else 'wdbc_fixed_40_v1'}
filename='wdbc_v2.joblib' if COMPARE else 'wdbc_v1.joblib'
joblib.dump(bundle,filename)
loaded=joblib.load(filename)

def predict_row(row):
    standard,pca,angle=loaded['steps']
    z=np.clip(angle.transform(pca.transform(standard.transform(row))),0,1)
    query_states,_=states(z)
    inputs={'quantum':gram(query_states,loaded['training_states']),'rbf':z}
    output={}
    for name,model in loaded['models'].items():
        v=inputs[name]
        record={'prediction':int(model.predict(v)[0]),'margin':float(model.decision_function(v)[0])}
        if COMPARE:
            target_index=int(np.flatnonzero(model.classes_==1)[0])
            record['class1_score']=float(model.predict_proba(v)[0,target_index])
        output[name]=record
    return output

# 저장 전후의 동일 표본 예측을 확인합니다.
expected=models['quantum'].predict(gram(states(b)[0],sa))
restored=np.array([predict_row(X[i:i+1])['quantum']['prediction'] for i in test])
assert np.array_equal(expected,restored)
print('model restore: predictions identical')
print('test quantum accuracy:',accuracy_score(y[test],restored))

app=Flask(__name__)
page='<!doctype html><html lang="ko"><meta charset="utf-8">\n<meta name="viewport" content="width=device-width,initial-scale=1">\n<title>WDBC 표본 분류</title><style>body{font-family:sans-serif;background:#edf2f8;padding:30px}main{max-width:760px;margin:auto;background:white;padding:28px;border-radius:12px}select,button{padding:12px;margin:8px 0}table{border-collapse:collapse;width:100%}td,th{padding:10px;border:1px solid #ccc}#result{white-space:pre-wrap;line-height:1.8}</style>\n<main><h1>WDBC 표본 분류 {{version}}</h1>\n<p>공개 세포 특징 데이터의 평가 표본을 선택합니다. 0은 원자료 양성, 1은 원자료 악성입니다. 임상적 사용에는 별도의 임상 검증이 필요합니다.</p>\n<label for="sample">원자료 행 번호</label><select id="sample">{% for n in samples %}<option value="{{n}}">표본 {{n}}</option>{% endfor %}</select>\n<button id="run">분류 실행</button><p id="status" role="status"></p><div id="result"></div>\n<script>document.getElementById(\'run\').onclick=async()=>{const button=document.getElementById(\'run\');button.disabled=true;const target=document.getElementById(\'result\');target.textContent=\'\';try{const response=await fetch(\'/predict\',{method:\'POST\',headers:{\'Content-Type\':\'application/json\'},body:JSON.stringify({source_row:Number(document.getElementById(\'sample\').value)})});const data=await response.json();if(!response.ok)throw new Error(data.error);let text=\'표본 \'+data.source_row+\' / 기준 분류 \'+data.reference+\'\\n\';text+=\'원본 입력: \'+JSON.stringify(data.features)+\'\\n\';for(const [name,result] of Object.entries(data.models)){text+=name+\': 예측 \'+result.prediction+\', 결정 점수 \'+result.margin.toFixed(4);if(result.class1_score!==undefined)text+=\', 클래스 1 보정 점수 \'+result.class1_score.toFixed(4);text+=\'\\n\';}target.textContent=text;document.getElementById(\'status\').textContent=\'분류 결과\';}catch(error){document.getElementById(\'status\').textContent=error.message;}finally{button.disabled=false;}};</script></main></html>'
valid={int(df.source_row.iloc[i]):int(i) for i in test}
@app.get('/')
def index():
    return render_template_string(page,samples=list(valid),version='v2.0' if COMPARE else 'v1.0')
@app.post('/predict')
def predict():
    payload=request.get_json(silent=True)
    value=payload.get('source_row') if isinstance(payload,dict) else None
    if type(value) is not int or value not in valid:
        return jsonify(error='목록의 평가 표본 번호를 선택해 주세요.'),400
    i=valid[value]
    return jsonify(source_row=value,reference=int(y[i]),
        features=dict(zip(FEATURES,X[i].tolist())),models=predict_row(X[i:i+1]))

client=app.test_client()
assert client.get('/').status_code==200
for row in valid:
    response=client.post('/predict',json={'source_row':row})
    assert response.status_code==200 and response.json['source_row']==row
for payload in [{},{'source_row':-1},{'source_row':True},{'source_row':'61'},[],None]:
    assert client.post('/predict',json=payload).status_code==400
print('web tests: 12 valid requests 200; invalid requests 400')
PORT=5002 if COMPARE else 5001
if 'server' in globals():
    server.shutdown()
server=make_server('0.0.0.0',PORT,app)
threading.Thread(target=server.serve_forever,daemon=True).start()
assert urlopen(f'http://127.0.0.1:{PORT}/').status==200
print('internal HTTP: 200')

# Google Colab에서 공용 주소로 접속합니다.
import platform, subprocess, re, time
from pathlib import Path
from urllib.request import urlretrieve
binary=Path('/tmp/cloudflared_day34')
architecture='arm64' if platform.machine()=='aarch64' else 'amd64'
if not binary.exists():
    urlretrieve(f'https://github.com/cloudflare/cloudflared/releases/download/2026.10.7/cloudflared-linux-{architecture}',binary)
    binary.chmod(0o755)
if 'tunnel' in globals() and tunnel.poll() is None:
    tunnel.terminate()
log_path=Path(f'tunnel_{PORT}.log')
log=log_path.open('w')
tunnel=subprocess.Popen([str(binary),'tunnel','--url',f'http://127.0.0.1:{PORT}',
    '--no-autoupdate','--protocol','http2'],stdout=log,stderr=log)
public_url=None
for _ in range(45):
    time.sleep(1)
    match=re.search(r'https://[a-z0-9]+(?:-[a-z0-9]+)+\.trycloudflare\.com',log_path.read_text())
    if match:
        public_url=match.group(0); break
    if tunnel.poll() is not None:
        break
if public_url:
    print('브라우저 접속 URL:',public_url)
else:
    print('공용 URL 생성 실패:',str(log_path),'연결 상태를 확인합니다.')
    tunnel.terminate()
# Colab 커널이 실행되는 동안 서버와 터널이 유지됩니다.

model restore: predictions identical
test quantum accuracy: 0.9166666666666666
web tests: 12 valid requests 200; invalid requests 400


INFO:werkzeug:127.0.0.1 - - [11/Nov/2026 13:08:48] "GET / HTTP/1.1" 200 -


internal HTTP: 200
브라우저 접속 URL: https://considering-rhythm-manufacturing-regime.trycloudflare.com


기본 분류 정확도는 두 모델 모두 11/12입니다. 같은 예측 클래스를 반환하더라도 결정 점수와 보정 점수는 달라질 수 있습니다. `class1_score`는 `classes_`에서 찾은 클래스 1의 보정 출력입니다. 이 수치를 환자 개인의 암 발생 확률로 사용하지 않습니다.

모델 저장·복원 후 평가 예측 일치, 표본 12개의 정상 API 요청과 잘못된 입력의 HTTP 400을 확인하는 코드가 실행됩니다. 내부 HTTP 200 출력은 서버 실행 상태를 확인합니다. 공용 URL로 접속하여 표본별 두 모델 결과가 함께 표시되는 화면을 기록합니다. 모델을 선택하려면 별도 검증 자료에서 오류 유형과 확률 보정 성능을 함께 평가합니다.

###